# Módulo 3: Agentes de IA
- **Sesión:** 02 — Frameworks de agentes: LangGraph y protocolo MCP
- **Laboratorio básico:** Primer servidor MCP
- **Fecha:** 28/10/2026

In [1]:
from langchain.tools import tool
from langchain.chat_models import init_chat_model

from dotenv import load_dotenv
import os
import logging

logging.getLogger("google_genai.models").setLevel(logging.ERROR)

from langchain.tools import tool



In [14]:
# Generar una funcion que obtenga el modelo LLM

def get_llm(prefix_model, temperature=0.9):
    """
    Obtiene un modelo de lenguaje (LLM) basado en el modelo especificado.

    Args:
        model (str): El nombre del modelo a utilizar.
        temperature (float, optional): La temperatura para la generación de texto. 
                                        Valores más altos producen respuestas más creativas.
                                        Valores más bajos producen respuestas más deterministas.
                                        Por defecto es 0.9.

    Returns:
        llm: Un objeto de modelo de lenguaje inicializado.
    """
    llm = init_chat_model(
                            model=prefix_model, 
                            # temperature=temperature
                          )
    return llm

# Configuración del modelo de lenguaje
load_dotenv(override = True)

prefix_model = os.getenv("PREFIX_MODEL")

# Obtención del modelo
llm = get_llm(prefix_model)


In [15]:
def ver_grafo(app):
    """Muestra el diagrama del grafo como imagen; si no hay conexión, lo imprime en formato Mermaid."""
    try:
        from IPython.display import Image, display
        display(Image(app.get_graph().draw_mermaid_png()))
    except Exception:
        print(app.get_graph().draw_mermaid())

def mostrar(mensajes):
    """Imprime el tipo y el contenido de cada mensaje."""
    for m in mensajes:
        if getattr(m, "tool_calls", None):
            texto = "solicita " + ", ".join(f"{t['name']}({t['args']})" for t in m.tool_calls)
        else:
            texto = m.text
        print(f"  [{type(m).__name__:12}] {texto[:120]}")

## Datos


In [16]:

PEDIDOS = {
    "48213": "en ruta, llega el 26/10/2026 a Arequipa",
    "77100": "entregado el 10/10/2026 en Lima",
    "50555": "en preparación, llega el 30/10/2026 a Cusco",
}

@tool
def consultar_pedido(numero_pedido: str) -> str:
    """Consulta el estado de un pedido de TiendaYa a partir de su número."""
    return PEDIDOS.get(numero_pedido, f"No existe el pedido {numero_pedido}.")

print(consultar_pedido.invoke({"numero_pedido": "48213"}))


en ruta, llega el 26/10/2026 a Arequipa


---
## MCP

**MCP (Model Context Protocol)** es un estándar para que una aplicación de IA use herramientas que viven **en otro programa**. Las herramientas se escriben una vez en un **servidor**, y cualquier aplicación compatible las puede usar.

| Término | Significado | En este notebook |
|---|---|---|
| **Servidor** | Programa que publica herramientas | `servidor_basico.py` |
| **Cliente** | Componente que se conecta al servidor y usa sus herramientas | `MultiServerMCPClient` |
| **Transporte stdio** | El cliente inicia el servidor y se comunican por la entrada y salida estándar | `"transport": "stdio"` |


---
## Ejercicio 1 — Un servidor con una herramienta

### Concepto
**FastMCP** convierte una función de Python en una herramienta MCP con el decorador `@mcp.tool()`. Igual que con `@tool`, el nombre, el docstring y los tipos se publican para que el modelo sepa cuándo y cómo usarla.

El servidor **no se ejecuta en el notebook**: la celda solo lo guarda como archivo (`%%writefile`). El cliente lo iniciará en el Ejercicio 2.

### Ejecutar

In [17]:
%%writefile servidor_basico.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("TiendaYa", log_level="ERROR")

PEDIDOS = {
    "48213": "en ruta, llega el 26/10/2026 a Arequipa",
    "50555": "en preparación, llega el 30/10/2026 a Cusco",
}

@mcp.tool()
def consultar_pedido(numero_pedido: str) -> str:
    """Consulta el estado de un pedido de TiendaYa a partir de su número."""
    return PEDIDOS.get(numero_pedido, f"No existe el pedido {numero_pedido}.")

if __name__ == "__main__":
    mcp.run(transport="stdio")

Overwriting servidor_basico.py


---
## Ejercicio 2 — Conectarse y listar las herramientas

### Concepto
`MultiServerMCPClient` es el cliente. Recibe la forma de iniciar cada servidor: el comando (`python`), el archivo y el transporte. `get_tools()` inicia el servidor, le pregunta qué herramientas tiene y las convierte en herramientas de LangChain.

Las operaciones MCP son **asíncronas**: en Jupyter se escribe `await` delante de la llamada.

### Ejecutar

In [7]:
import sys
from pathlib import Path
from langchain_mcp_adapters.client import MultiServerMCPClient

CONEXIONES = {
    "tiendaya": {
        "command": sys.executable,                          # el Python del notebook
        "args": [str(Path("servidor_basico.py").resolve())],
        "transport": "stdio",
    }
}

cliente = MultiServerMCPClient(CONEXIONES)
herramientas = await cliente.get_tools()

for h in herramientas:
    print(f"{h.name}: {h.description}")

consultar_pedido: Consulta el estado de un pedido de TiendaYa a partir de su número.


---
## Ejercicio 3 — Usar la herramienta sin el modelo

### Concepto
Una herramienta MCP se puede invocar directamente, como cualquier herramienta de LangChain. Como es asíncrona, se usa `ainvoke` con `await`. El resultado llega como una lista de bloques de contenido; el texto está en la clave `"text"`.

### Ejecutar

In [8]:
consultar = herramientas[0]
resultado = await consultar.ainvoke({"numero_pedido": "48213"})

print("Resultado completo:", resultado)
print("Texto:", resultado[0]["text"])

Resultado completo: [{'type': 'text', 'text': 'en ruta, llega el 26/10/2026 a Arequipa', 'id': 'lc_9996d46c-261e-4a72-90d1-6507177c91bc'}]
Texto: en ruta, llega el 26/10/2026 a Arequipa


### Práctica
Consultar el pedido `99999` y verificar que el servidor responde que no existe.

---
## Ejercicio 4 — Publicar una herramienta nueva

### Concepto
Se agrega al servidor la herramienta `consultar_horario`. Basta volver a escribir el archivo; el cliente la descubre en la siguiente llamada a `get_tools()`.

### Ejecutar

In [10]:
%%writefile servidor_basico.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("TiendaYa", log_level="ERROR")

PEDIDOS = {
    "48213": "en ruta, llega el 26/10/2026 a Arequipa",
    "50555": "en preparación, llega el 30/10/2026 a Cusco",
}


@mcp.tool()
def consultar_pedido(numero_pedido: str) -> str:
    """Consulta el estado de un pedido de TiendaYa a partir de su número."""
    return PEDIDOS.get(numero_pedido, f"No existe el pedido {numero_pedido}.")


@mcp.tool()
def consultar_horario(dia: str) -> str:
    """Devuelve el horario de atención de TiendaYa para un día de la semana."""
    if dia.lower() in ["sabado", "sábado", "domingo"]:
        return "Fin de semana: de 10:00 a 14:00."
    return "Lunes a viernes: de 8:00 a 20:00."


if __name__ == "__main__":
    mcp.run(transport="stdio")

Overwriting servidor_basico.py


In [11]:
herramientas = await cliente.get_tools()          # se vuelven a descubrir

for h in herramientas:
    print(f"{h.name}: {h.description}")

consultar_pedido: Consulta el estado de un pedido de TiendaYa a partir de su número.
consultar_horario: Devuelve el horario de atención de TiendaYa para un día de la semana.


---
## Ejercicio 5 — Un agente con las herramientas del servidor

### Concepto
Las herramientas obtenidas con `get_tools()` se entregan a `create_agent` igual que las herramientas definidas con `@tool`. El agente **no sabe** que viven en otro programa.

```
agente ──▶ cliente MCP ──(stdio)──▶ servidor_basico.py
```

Como las herramientas son asíncronas, el agente se ejecuta con `await agente.ainvoke(...)`.

### Ejecutar

In [13]:
from langchain.agents import create_agent

agente = create_agent(
    model=llm,
    tools=herramientas,                      # herramientas del servidor MCP
    system_prompt="Eres el asistente de TiendaYa. Usa las herramientas para obtener los datos; no los inventes.",
)

for consulta in ["¿Dónde está mi pedido 50555?", "¿A qué hora atienden el domingo?"]:
    r = await agente.ainvoke({"messages": [("user", consulta)]})
    usadas = [t["name"] for m in r["messages"] for t in getattr(m, "tool_calls", [])]
    print(f"{consulta}\n  herramientas usadas: {usadas}\n  respuesta: {r['messages'][-1].text}\n")

¿Dónde está mi pedido 50555?
  herramientas usadas: ['consultar_pedido']
  respuesta: El estado de tu pedido 50555 es **en preparación**, y está programado para llegar el **30/10/2026** a Cusco.

¿A qué hora atienden el domingo?
  herramientas usadas: ['consultar_horario']
  respuesta: El horario de atención para el domingo es de 10:00 a 14:00.

